# 🔬 UMPA-SAM Training on Google Colab

Train the UMPA-SAM model with `CascadedPromptFusionEncoder` on 5 polyp benchmark datasets.

**Architecture**: `sparse_prompt_embeddings = e_fused.unsqueeze(1)`

**Requirements**: GPU runtime (T4 15GB recommended)

---

## 1. Setup Environment

In [ ]:
# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Clone repo (branch tmf-umpa)
!git clone -b tmf-umpa https://github.com/TongDuyDat/UMPA-SAM.git /content/sam3
%cd /content/sam3

In [ ]:
# Install dependencies
!pip install -q einops ftfy regex iopath huggingface-hub \
    opencv-python albumentations tqdm \
    matplotlib pandas scikit-learn scikit-image \
    pycocotools tensorboard torchmetrics pydantic pyyaml

## 2. Load SAM3 Checkpoint

Upload `sam3.pt` từ Google Drive (checkpoint riêng, không có trên HuggingFace)

In [ ]:
import os
os.makedirs('model_trained', exist_ok=True)

SAM3_CHECKPOINT = 'model_trained/sam3.pt'

# --- Option A: Copy từ Google Drive (RECOMMENDED) ---
from google.colab import drive
drive.mount('/content/drive')

# ⚠️ Sửa đường dẫn Drive cho đúng vị trí bạn lưu sam3.pt
DRIVE_CHECKPOINT = '/content/drive/MyDrive/UMPA-SAM/sam3.pt'

if os.path.exists(DRIVE_CHECKPOINT):
    !cp "{DRIVE_CHECKPOINT}" model_trained/sam3.pt
    print(f'✅ Copied sam3.pt from Drive')
else:
    print(f'❌ Không tìm thấy: {DRIVE_CHECKPOINT}')
    print('Hãy upload sam3.pt lên Drive hoặc dùng Option B bên dưới')

# --- Option B: Upload trực tiếp từ máy ---
# from google.colab import files
# uploaded = files.upload()  # Chọn sam3.pt
# !mv sam3.pt model_trained/sam3.pt

if os.path.exists(SAM3_CHECKPOINT):
    print(f'Size: {os.path.getsize(SAM3_CHECKPOINT) / 1e6:.1f} MB')
else:
    raise FileNotFoundError(f'{SAM3_CHECKPOINT} not found!')

## 3. Download Polyp Datasets

Download 5 benchmark datasets vào `umpt_sam/data/`

In [ ]:
# ================================================================
# Upload datasets từ Google Drive hoặc download từ link
# ================================================================

# --- Option A: Copy từ Drive (nếu bạn đã upload datasets lên Drive) ---
# !cp -r /content/drive/MyDrive/polyp_datasets/* umpt_sam/data/

# --- Option B: Download từ public links ---
# Kvasir-SEG
# !wget -q https://datasets.simula.no/downloads/kvasir-seg.zip -O /tmp/kvasir.zip
# !unzip -q /tmp/kvasir.zip -d /tmp/kvasir
# !mv /tmp/kvasir/Kvasir-SEG umpt_sam/data/Kvasir_SEG

# --- Option C: Upload zip trực tiếp ---
# from google.colab import files
# uploaded = files.upload()  # upload zip file
# !unzip -q *.zip -d umpt_sam/data/

print('\n📂 Dataset directories:')
!ls -la umpt_sam/data/ | grep -E "^d"

In [ ]:
# Verify datasets & auto-generate splits
import sys
sys.path.insert(0, '/content/sam3')

from umpt_sam.data.dataset_registry import list_datasets, ensure_all_splits

print('Registered datasets:', list_datasets())
ensure_all_splits()

# Quick count
for ds in list_datasets():
    img_dir = f'umpt_sam/data/{ds}/images'
    if os.path.isdir(img_dir):
        n = len(os.listdir(img_dir))
        print(f'  ✅ {ds}: {n} images')
    else:
        print(f'  ❌ {ds}: NOT FOUND')

## 4. Training Configuration

In [ ]:
# ================================================================
# Training Hyperparameters — EDIT HERE
# ================================================================

TRAIN_CONFIG = {
    # Model
    'sam_checkpoint': 'model_trained/sam3.pt',
    'embed_dim': 256,
    'text_embed_dim': 512,
    
    # Training
    'batch_size': 4,       # T4: 4, A100: 16-32
    'K': 3,                # Number of perturbations
    'device': 'cuda',
    
    # 3-phase schedule
    'phase1': {'epochs': 5,  'lr': 1e-4, 'lambda_con': 0.0},
    'phase2': {'epochs': 5,  'lr': 5e-5, 'lambda_con': 0.0},
    'phase3': {'epochs': 10, 'lr': 1e-5, 'lambda_con': 0.5},
}

print(f"Total epochs: {sum(p['epochs'] for p in [TRAIN_CONFIG['phase1'], TRAIN_CONFIG['phase2'], TRAIN_CONFIG['phase3']])}")
print(f"Batch size: {TRAIN_CONFIG['batch_size']}")

## 5. Train on Single Dataset

In [ ]:
import gc
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

from umpt_sam.umpa_model import UMPAModel
from umpt_sam.config.model_config import UMPAModelConfig, UPFEConfig, MPPGConfig
from umpt_sam.config.train_config import TrainConfig, PhaseConfig
from umpt_sam.losses.composer_loss import ComposerLoss
from umpt_sam.training.phase_scheduler import PhaseScheduler
from umpt_sam.training.trainer import UMPATrainer
from umpt_sam.training.evaluate import evaluate
from umpt_sam.data.polyp_dataset import PolypDataset, DatasetConfig, collate_fn
from umpt_sam.data.dataset_registry import get_dataset_config
from umpt_sam.data.polyp_transforms import POLYP_TRANSFORMS


def train_single_dataset(
    dataset_name: str,
    cfg: dict,
    save_dir: str = 'checkpoints',
):
    device = cfg['device']
    print(f"\n{'='*60}")
    print(f"🚀 Training on: {dataset_name}")
    print(f"{'='*60}")

    # --- Data ---
    dataset_config = get_dataset_config(dataset_name)
    
    train_dataset = PolypDataset(cfg=dataset_config, phase='train')
    train_dataset.transform = POLYP_TRANSFORMS['train']
    
    val_dataset = PolypDataset(cfg=dataset_config, phase='val')
    val_dataset.transform = POLYP_TRANSFORMS['val']
    
    test_dataset = PolypDataset(cfg=dataset_config, phase='test')
    test_dataset.transform = POLYP_TRANSFORMS['val']
    
    bs = cfg['batch_size']
    train_loader = DataLoader(train_dataset, batch_size=bs, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=bs, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=bs, shuffle=False, collate_fn=collate_fn)
    
    print(f'📦 Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}')

    # --- Model ---
    model_config = UMPAModelConfig(
        sam_checkpoint=cfg['sam_checkpoint'],
        embed_dim=cfg['embed_dim'],
        text_embed_dim=cfg['text_embed_dim'],
        freeze_image_encoder=True,
        upfe=UPFEConfig(scoring_hidden_dim=cfg['embed_dim']),
        mppg=MPPGConfig(),
    )
    model = UMPAModel.from_config(model_config=model_config).to(device)
    
    total_p = sum(p.numel() for p in model.parameters())
    trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'📊 Params: {total_p/1e6:.1f}M total, {trainable_p/1e6:.1f}M trainable')

    # --- Training config ---
    p1, p2, p3 = cfg['phase1'], cfg['phase2'], cfg['phase3']
    
    train_config = TrainConfig(
        batch_size=bs,
        K=cfg['K'],
        phase1=PhaseConfig(
            name='warmup', epochs=p1['epochs'], lr=p1['lr'],
            lambda_con=p1['lambda_con'],
            freeze_image_encoder=True, freeze_prompt_encoder=True, freeze_mask_decoder=True,
        ),
        phase2=PhaseConfig(
            name='adaptation', epochs=p2['epochs'], lr=p2['lr'],
            lambda_con=p2['lambda_con'],
            freeze_image_encoder=True, freeze_prompt_encoder=False, freeze_mask_decoder=True,
        ),
        phase3=PhaseConfig(
            name='consistency', epochs=p3['epochs'], lr=p3['lr'],
            lambda_con=p3['lambda_con'],
            freeze_image_encoder=True, freeze_prompt_encoder=False, freeze_mask_decoder=False,
        ),
    )

    # --- Loss + Optimizer ---
    composer_loss = ComposerLoss(config_loss=train_config.loss_weights).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=train_config.phase1.lr, weight_decay=1e-4)
    scheduler = PhaseScheduler(train_config=train_config)

    # --- Trainer ---
    save_path = os.path.join(save_dir, dataset_name)
    trainer = UMPATrainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        train_config=train_config,
        composer_loss=composer_loss,
        evaluate_fn=evaluate,
        save_dir=save_path,
        device=device,
    )

    trainer.run()
    print(f'\n✅ Done: {dataset_name} | Best Dice: {trainer.best_val_dice:.4f}')
    print(f'📁 Saved to: {trainer.save_dir}')
    
    return trainer

In [ ]:
# ================================================================
# 🚀 START TRAINING — Single dataset
# ================================================================

trainer = train_single_dataset(
    dataset_name='kvasir_seg',  # Change: cvc_clinicdb, cvc_colondb, cvc_300, etis_larib
    cfg=TRAIN_CONFIG,
)

## 6. Train on ALL 5 Datasets (Sequential)

In [ ]:
import time
from umpt_sam.data.dataset_registry import list_datasets

all_results = {}
start_total = time.time()

for i, ds_name in enumerate(list_datasets(), 1):
    print(f"\n{'█'*60}")
    print(f"█ [{i}/{len(list_datasets())}] Dataset: {ds_name}")
    print(f"{'█'*60}")
    
    try:
        t = train_single_dataset(ds_name, TRAIN_CONFIG)
        all_results[ds_name] = {
            'status': 'OK',
            'best_dice': t.best_val_dice,
            'save_dir': t.save_dir,
        }
    except Exception as e:
        print(f'❌ FAILED: {ds_name} — {e}')
        all_results[ds_name] = {'status': 'FAILED', 'error': str(e)}
    finally:
        gc.collect()
        torch.cuda.empty_cache()

elapsed = time.time() - start_total
h, rem = divmod(int(elapsed), 3600)
m, s = divmod(rem, 60)

print(f"\n{'='*60}")
print(f'📊 RESULTS SUMMARY — {h}h {m}m {s}s')
print(f"{'='*60}")
for ds, r in all_results.items():
    if r['status'] == 'OK':
        print(f"  ✅ {ds:20s} | Dice: {r['best_dice']:.4f}")
    else:
        print(f"  ❌ {ds:20s} | Error: {r['error'][:80]}")

## 7. Save Results to Google Drive

In [ ]:
DRIVE_SAVE_DIR = '/content/drive/MyDrive/UMPA-SAM/checkpoints'
!mkdir -p "{DRIVE_SAVE_DIR}"
!cp -r checkpoints/* "{DRIVE_SAVE_DIR}/"
print(f'\n✅ Checkpoints saved to: {DRIVE_SAVE_DIR}')
!ls -la "{DRIVE_SAVE_DIR}/"

## 8. Quick Evaluation

In [ ]:
EVAL_DATASET = 'kvasir_seg'
CHECKPOINT = 'checkpoints/kvasir_seg/best_model.pth'  # Update path

model_config = UMPAModelConfig(
    sam_checkpoint='model_trained/sam3.pt',
    embed_dim=256, text_embed_dim=512,
    freeze_image_encoder=True,
    upfe=UPFEConfig(scoring_hidden_dim=256),
    mppg=MPPGConfig(),
)
model = UMPAModel.from_config(model_config=model_config).to('cuda')

ckpt = torch.load(CHECKPOINT, map_location='cuda')
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded from epoch {ckpt.get('epoch', '?')}")

dataset_config = get_dataset_config(EVAL_DATASET)
test_dataset = PolypDataset(cfg=dataset_config, phase='test')
test_dataset.transform = POLYP_TRANSFORMS['val']
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)

metrics = evaluate(model, test_loader, device='cuda')
print(f'\n📊 Test Results ({EVAL_DATASET}):')
for k, v in metrics.items():
    print(f'   {k}: {v:.4f}')